# Link Budget Verification

Compares the analytical per-block power budget against the simulated power at each tap of the RF pipeline, plus a sweep over input power.

In [9]:
import numpy as np
import matplotlib.pyplot as plt

from rfmodel.core.units import dbm_to_w
from rfmodel.plot_utils.plot_block_diagram import plot_pipeline

from tx_setup import build_test_setup

## Setup

In [10]:
setup = build_test_setup(input_pwr_dbm=-30.0)

pipe                = setup.pipe
cfg                 = setup.cfg
sig_ofdm            = setup.sig_ofdm
sig_ofdm_normalized = setup.sig_ofdm_normalized
input_pwr_dbm       = setup.input_pwr_dbm

plot_pipeline(pipe)

KeyError: 'snr_db'

## Analytical link budget

In [ ]:
blocks = {b['name']: b['params'] for b in cfg['pipeline']}
c0     = 299792458.0

G_mix_tx = blocks['mixer_and_pll_TX']['gain_db']
G_pa     = blocks['PA_TX']['gain_db']
G_lna    = blocks['LNA']['gain_db']
G_mix_rx = blocks['mixer_and_pll_RX']['gain_db']

p      = blocks['PathLoss']
lam    = c0 / float(p['freq_hz'])
L_path = 10 * np.log10((lam / (4 * np.pi * p['distance_m']))**2) \
         + p['tx_ant_gain_db'] + p['rx_ant_gain_db']

P_in   = input_pwr_dbm
P_mix1 = P_in   + G_mix_tx
P_pa   = P_mix1 + G_pa
P_path = P_pa   + L_path
P_awgn = P_path + 0.0
P_end  = P_awgn

print('P_in      =', P_in,   'dBm')
print('Mixer TX  =', P_mix1, 'dBm')
print('PA_TX     =', P_pa,   'dBm')
print('PathLoss  =', P_path, 'dBm')
print('Prx       =', P_end,  'dBm')

P_in      = -30.0 dBm
Mixer TX  = -30.0 dBm
PA_TX     = -10.0 dBm
PathLoss  = -58.14778322188338 dBm
Prx       = -58.14778322188338 dBm


## Simulated power per tap

In [ ]:
def p_dbm(x):
    return 10 * np.log10(np.mean(np.abs(x) ** 2)) + 30

tap_names = ['mixer_and_pll_TX', 'PA_TX', 'PathLoss', 'AWGN', 'LNA', 'mixer_and_pll_RX']
_, taps   = pipe.run(sig_ofdm_normalized, taps=tap_names)

print('P_in      =', p_dbm(sig_ofdm_normalized.x),     'dBm')
print('Mixer TX  =', p_dbm(taps['mixer_and_pll_TX'].x), 'dBm')
print('PA_TX     =', p_dbm(taps['PA_TX'].x),            'dBm')
print('PathLoss  =', p_dbm(taps['PathLoss'].x),         'dBm')
print('AWGN      =', p_dbm(taps['AWGN'].x),             'dBm')
print('P_rx      =', p_dbm(taps['AWGN'].x),             'dBm')

P_in      = -30.0 dBm
Mixer TX  = -30.000068607567336 dBm
PA_TX     = -10.00570856413568 dBm
PathLoss  = -58.15349178601906 dBm
AWGN      = -58.14344243725384 dBm
P_rx      = -58.14344243725384 dBm


## Analytical vs simulated — summary

In [ ]:
rows = [
    ('P_in',     P_in,   p_dbm(sig_ofdm_normalized.x)),
    ('Mixer TX', P_mix1, p_dbm(taps['mixer_and_pll_TX'].x)),
    ('PA_TX',    P_pa,   p_dbm(taps['PA_TX'].x)),
    ('PathLoss', P_path, p_dbm(taps['PathLoss'].x)),
    ('AWGN',     P_path, p_dbm(taps['AWGN'].x)),
    ('P_rx',     P_end,  p_dbm(taps['AWGN'].x)),
]

print('Stage      P_an [dBm]   P_sim [dBm]   Δ [dB]')
for name, an, sim in rows:
    print(f'{name:<10} {an:>10.2f}   {sim:>10.2f}   {sim - an:>+7.2f}')

Stage      P_an [dBm]   P_sim [dBm]   Δ [dB]
P_in           -30.00       -30.00     +0.00
Mixer TX       -30.00       -30.00     -0.00
PA_TX          -10.00       -10.01     -0.01
PathLoss       -58.15       -58.15     -0.01
AWGN           -58.15       -58.14     +0.00
P_rx           -58.15       -58.14     +0.00


## Sweep over input power

In [ ]:
# Sweep input power and compare measured RX power against the analytical line.
p_in_sweep_dbm = np.arange(-50.0, 15.0, 2.0)

G_tot_db  = P_end - P_in
current_w = np.mean(np.abs(sig_ofdm.x) ** 2)

p_rx_sim = []
p_rx_an  = []

for pin in p_in_sweep_dbm:
    scale = np.sqrt(dbm_to_w(pin) / current_w)
    s_in  = sig_ofdm.copy_with(x=sig_ofdm.x * scale)
    _, taps_sw = pipe.run(s_in, taps=['AWGN'])
    p_rx_sim.append(10 * np.log10(np.mean(np.abs(taps_sw['AWGN'].x) ** 2)) + 30)
    p_rx_an.append(pin + G_tot_db)

p_rx_sim = np.array(p_rx_sim)
p_rx_an  = np.array(p_rx_an)

plt.figure(figsize=(7, 4.5))
plt.plot(p_in_sweep_dbm, p_rx_an,  '--', label='Analytical')
plt.plot(p_in_sweep_dbm, p_rx_sim, 'o-', markersize=4, label='Simulated')
plt.xlabel('P_in (dBm)')
plt.ylabel('P_rx (dBm)')
plt.title(f"Link budget sweep (PA P1dB @ {blocks['PA_TX']['p1db_out_dbm']} dBm)")
plt.grid(True)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()